### Public Tool （RMSNorm / SwiGLU）

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F


# ========= Shared utilities =========

class RMSNorm(nn.Module):
    """RMSNorm: same formula as Flax版."""
    def __init__(self, dim, eps=1e-8):
        super().__init__()
        self.eps = eps
        self.scale = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        # x: (..., D)
        rms = x.pow(2).mean(dim=-1, keepdim=True).add(self.eps).sqrt()
        return x * (self.scale / rms)


class SwiGLU(nn.Module):
    """SwiGLU: 与 Flax 版结构对应：u,v 两个投影 → silu(u)*v → 投回 d_model."""
    def __init__(self, d_model, mult=2.667, dropout=0.1):
        super().__init__()
        hidden = int(d_model * mult)
        self.w_u = nn.Linear(d_model, hidden, bias=False)
        self.w_v = nn.Linear(d_model, hidden, bias=False)
        self.proj = nn.Linear(hidden, d_model, bias=False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        u = self.w_u(x)
        v = self.w_v(x)
        x = F.silu(u) * v
        x = self.dropout(x)
        x = self.proj(x)
        return x


### RoPE + Multi-Query Attention

In [ ]:
# ========= RoPE + Multi-query attention =========

def rotary_emb(d: int, T: int, base: float = 10000.0, device=None, dtype=torch.float32):
    """
    PyTorch 版 rotary_emb：返回 cos, sin 形状为 (1, 1, T, d/2)
    """
    assert d % 2 == 0
    device = device or torch.device("cpu")
    inv_freq = 1.0 / (base ** (torch.arange(0, d, 2, device=device, dtype=dtype) / d))
    t = torch.arange(T, device=device, dtype=dtype)
    freqs = torch.einsum("t,f->tf", t, inv_freq)
    cos = freqs.cos()[None, None, ...]  # (1,1,T,d/2)
    sin = freqs.sin()[None, None, ...]
    return cos, sin


def apply_rope(x, cos, sin):
    """
    x: (B, H, T, Dh), Dh 偶数
    cos,sin: (1,1,T,Dh/2)
    """
    Dh = x.size(-1)
    assert Dh % 2 == 0
    x1, x2 = torch.split(x, Dh // 2, dim=-1)
    cos = cos[..., :x1.size(-2), :]  # 对齐 T
    sin = sin[..., :x1.size(-2), :]
    xc = x1 * cos - x2 * sin
    xs = x1 * sin + x2 * cos
    return torch.cat([xc, xs], dim=-1)


class MQSelfAttentionPT(nn.Module):
    """
    Multi-query self-attention + RoPE，对应 Flax MQSelfAttention。
    - Q: per head, shape (B, H, T, Dh)
    - K,V: 按 n_kv_heads 建，再重复到 n_heads
    - mask: 期望形状 (B, 1, T, T)，bool
    """
    def __init__(self, d_model, n_heads, n_kv_heads=1, dropout=0.1, max_len=2048):
        super().__init__()
        assert d_model % n_heads == 0
        self.d_model = d_model
        self.n_heads = n_heads
        self.n_kv_heads = n_kv_heads
        self.head_dim = d_model // n_heads
        assert self.head_dim % 2 == 0, "head_dim must be even for RoPE"

        self.q_proj = nn.Linear(d_model, d_model, bias=False)
        self.k_proj = nn.Linear(d_model, self.head_dim * n_kv_heads, bias=False)
        self.v_proj = nn.Linear(d_model, self.head_dim * n_kv_heads, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)
        self.dropout = nn.Dropout(dropout)

        cos, sin = rotary_emb(self.head_dim, max_len)
        self.register_buffer("rope_cos", cos, persistent=False)
        self.register_buffer("rope_sin", sin, persistent=False)

    def forward(self, x, mask=None):
        """
        x: (B, T, D)
        mask: (B, 1, T, T) bool 或 None
        """
        B, T, D = x.shape
        H, H_kv, Dh = self.n_heads, self.n_kv_heads, self.head_dim

        q = self.q_proj(x)                                # (B, T, D)
        k = self.k_proj(x)                                # (B, T, H_kv*Dh)
        v = self.v_proj(x)

        q = q.view(B, T, H, Dh).permute(0, 2, 1, 3)       # (B,H,T,Dh)
        k = k.view(B, T, H_kv, Dh).permute(0, 2, 1, 3)    # (B,H_kv,T,Dh)
        v = v.view(B, T, H_kv, Dh).permute(0, 2, 1, 3)

        cos = self.rope_cos[:, :, :T, :]
        sin = self.rope_sin[:, :, :T, :]
        q = apply_rope(q, cos, sin)
        k = apply_rope(k, cos, sin)

        if H_kv != H:
            factor = H // H_kv
            k = k.repeat(1, factor, 1, 1)   # (B,H,T,Dh)
            v = v.repeat(1, factor, 1, 1)
        else:
            # (B,H_kv,T,Dh) → (B,H,T,Dh)
            pass

        # 注意：此时 k,v shape 也为 (B,H,T,Dh)
        att = torch.einsum("bhtd,bhTd->bhtT", q, k) / math.sqrt(Dh)  # (B,H,T,T)

        if mask is not None:
            # mask: True 表示可见，False 表示屏蔽
            # 需要广播到 (B,H,T,T)
            if mask.ndim == 4:
                att = att.masked_fill(~mask, float("-inf"))
            else:
                raise ValueError("mask 应该是 (B,1,T,T) 形式")

        att = F.softmax(att, dim=-1)
        att = self.dropout(att)

        y = torch.einsum("bhtT,bhTd->bhtd", att, v)       # (B,H,T,Dh)
        y = y.permute(0, 2, 1, 3).contiguous().view(B, T, D)
        y = self.out_proj(y)
        y = self.dropout(y)
        return y


class DecoderBlockMQPT(nn.Module):
    """
    对应 Flax DecoderBlock (RoPE + MQSelfAttention + SwiGLU).
    """
    def __init__(self, d_model, n_heads, n_kv_heads=1,
                 attn_dropout=0.1, mlp_dropout=0.1, resid_scale_init=1e-2, max_len=2048):
        super().__init__()
        self.d_model = d_model
        self.res_scale = nn.Parameter(torch.full((), resid_scale_init))
        self.norm1 = RMSNorm(d_model)
        self.norm2 = RMSNorm(d_model)
        self.attn = MQSelfAttentionPT(d_model, n_heads, n_kv_heads, dropout=attn_dropout, max_len=max_len)
        self.mlp = SwiGLU(d_model, mult=2.667, dropout=mlp_dropout)

    def forward(self, x, mask=None):
        h = self.norm1(x)
        h = self.attn(h, mask=mask)
        x = x + self.res_scale * h

        h = self.norm2(x)
        h = self.mlp(h)
        x = x + self.res_scale * h
        return x


class DecoderOnlyTransformerMQPT(nn.Module):
    """
    PyTorch 版 DecoderOnlyTransformer:
    - vocab embedding
    - n 层 DecoderBlockMQPT
    - final RMSNorm + tied softmax

    forward(idx, targets=None, pad_mask=None) → (logits, loss)
    """
    def __init__(self, vocab_size, d_model, n_layers, n_heads,
                 n_kv_heads=1, max_len=2048, dropout=0.1):
        super().__init__()
        self.vocab_size = vocab_size
        self.d_model = d_model
        self.n_layers = n_layers
        self.max_len = max_len

        self.tok_embed = nn.Embedding(vocab_size, d_model)
        self.dropout = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([
            DecoderBlockMQPT(d_model, n_heads, n_kv_heads,
                             attn_dropout=dropout, mlp_dropout=dropout,
                             resid_scale_init=1e-2, max_len=max_len)
            for _ in range(n_layers)
        ])
        self.final_norm = RMSNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        self.lm_head.weight = self.tok_embed.weight

    def _build_causal_mask(self, B, T, device):
        mask = torch.tril(torch.ones(T, T, dtype=torch.bool, device=device))  # (T,T)
        mask = mask.view(1, 1, T, T).expand(B, 1, T, T)                      # (B,1,T,T)
        return mask

    def forward(self, idx, targets=None, pad_mask=None):
        """
        idx: (B,T) long
        pad_mask: (B,T) bool, True=keep, False=pad；目前先不细做 combine，与 Flax 略有差别。
        """
        B, T = idx.shape
        assert T <= self.max_len
        device = idx.device

        x = self.tok_embed(idx)               # (B,T,D)
        x = self.dropout(x)

        mask = self._build_causal_mask(B, T, device)

        for blk in self.blocks:
            x = blk(x, mask=mask)

        x = self.final_norm(x)
        logits = torch.einsum("btd,vd->btv", x, self.tok_embed.weight)  # weight tying

        loss = None
        if targets is not None:
            loss = F.cross_entropy(
                logits.view(-1, logits.size(-1)),
                targets.view(-1)
            )
        return logits, loss


### Transformer-XL with Memory

In [ ]:
# ========= Transformer-XL style memory =========

class XLBlockPT(nn.Module):
    """
    对应 Flax XLBlock：
    - Q 从当前 x 得到
    - K,V 从 concat(mem, x) 的 RMSNorm(cat) 得到
    - mask 允许看所有 memory + 当前 segment 的历史。
    """
    def __init__(self, d_model, n_heads,
                 mlp_mult=2.667,
                 attn_dropout=0.1,
                 mlp_dropout=0.1,
                 resid_scale_init=1e-2,
                 mem_len=512):
        super().__init__()
        self.d_model = d_model
        self.n_heads = n_heads
        self.mlp_mult = mlp_mult
        self.mem_len = mem_len
        self.head_dim = d_model // n_heads

        self.norm_q = RMSNorm(d_model)
        self.norm_kv = RMSNorm(d_model)

        self.q_proj = nn.Linear(d_model, d_model, bias=False)
        self.kv_proj = nn.Linear(d_model, 2 * d_model, bias=False)
        self.o_proj = nn.Linear(d_model, d_model, bias=False)

        self.attn_drop = nn.Dropout(attn_dropout)
        self.mlp = SwiGLU(d_model, mult=mlp_mult, dropout=mlp_dropout)
        self.res_scale = nn.Parameter(torch.full((), resid_scale_init))

    def forward(self, x, mem=None):
        """
        x: (B,T,D)
        mem: (B,M,D) or None
        return: x_out, new_mem
        """
        B, T, D = x.shape
        if mem is None:
            mem = torch.zeros(B, 0, D, dtype=x.dtype, device=x.device)
        cat = torch.cat([mem, x], dim=1)      # (B, M+T, D)
        M = mem.size(1)

        # attention
        h_q = self.norm_q(x)
        h_kv = self.norm_kv(cat)

        q = self.q_proj(h_q)                  # (B,T,D)
        kv = self.kv_proj(h_kv)               # (B,M+T,2D)
        k, v = torch.chunk(kv, 2, dim=-1)     # each (B,M+T,D)

        H = self.n_heads
        Dh = self.head_dim

        q = q.view(B, T, H, Dh).permute(0, 2, 1, 3)      # (B,H,T,Dh)
        k = k.view(B, M + T, H, Dh).permute(0, 2, 1, 3)  # (B,H,M+T,Dh)
        v = v.view(B, M + T, H, Dh).permute(0, 2, 1, 3)

        # mask: 前 M 是 memory，全可见；后 T 是当前 segment，因果。
        base = torch.ones(1, 1, T, M + T, dtype=torch.bool, device=x.device)
        tri = torch.tril(torch.ones(T, T, dtype=torch.bool, device=x.device))
        base[:, :, :, M:] = tri[None, None, :, :]   # (1,1,T,M+T)
        mask = base

        att_logits = torch.einsum("bhtd,bhTd->bhtT", q, k) / math.sqrt(Dh)  # (B,H,T,M+T)
        att_logits = att_logits.masked_fill(~mask, float("-inf"))
        att_probs = F.softmax(att_logits, dim=-1)
        att_probs = self.attn_drop(att_probs)

        y = torch.einsum("bhtT,bhTd->bhtd", att_probs, v)   # (B,H,T,Dh)
        y = y.permute(0, 2, 1, 3).contiguous().view(B, T, D)
        y = self.o_proj(y)

        # 第一层残差
        x = x + self.res_scale * y

        # MLP 残差
        h = RMSNorm(self.d_model)(x)
        y = self.mlp(h)
        x = x + self.res_scale * y

        # 更新 memory：拼 mem + x，取最后 mem_len
        new_mem = torch.cat([mem, x], dim=1)[:, -self.mem_len:, :].detach()
        return x, new_mem


class XLMemoryTransformerPT(nn.Module):
    """
    PyTorch 版 XLMemoryTransformer：
    - forward(idx, mems=None, targets=None) → logits, loss, new_mems
    """
    def __init__(self, vocab_size, d_model, n_layers, n_heads,
                 mem_len=512, dropout=0.1, mlp_mult=2.667):
        super().__init__()
        self.vocab_size = vocab_size
        self.d_model = d_model
        self.n_layers = n_layers
        self.mem_len = mem_len

        self.tok_embed = nn.Embedding(vocab_size, d_model)
        self.dropout = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([
            XLBlockPT(d_model, n_heads,
                     mlp_mult=mlp_mult,
                     attn_dropout=dropout,
                     mlp_dropout=dropout,
                     resid_scale_init=1e-2,
                     mem_len=mem_len)
            for _ in range(n_layers)
        ])
        self.final_norm = RMSNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        self.lm_head.weight = self.tok_embed.weight

    def forward(self, idx, mems=None, targets=None):
        """
        idx: (B,T)
        mems: list of length n_layers, each (B,M,D) 或 None
        返回: logits, loss, new_mems
        """
        B, T = idx.shape
        x = self.tok_embed(idx)
        x = self.dropout(x)

        if mems is None:
            mems = [None] * self.n_layers

        new_mems = []
        for blk, mem in zip(self.blocks, mems):
            x, mem_out = blk(x, mem=mem)
            new_mems.append(mem_out)

        x = self.final_norm(x)
        logits = torch.einsum("btd,vd->btv", x, self.tok_embed.weight)

        loss = None
        if targets is not None:
            loss = F.cross_entropy(
                logits.view(-1, logits.size(-1)),
                targets.view(-1)
            )
        return logits, loss, new_mems


### Switch / Top-1 MoE

In [ ]:
# ========= Switch / Top-1 MoE =========

class SwitchMoEPT(nn.Module):
    """
    对应 Flax SwitchMoE:
    - router_w: (D, n_experts)
    - logits = x @ router_w
    - top1 expert 分配
    - 每个 expert: Dense(H) -> silu -> Dense(D)
    - 返回: out, aux_loss (expert usage + balance)
    """
    def __init__(self, d_model, mult=2.667, n_experts=8, dropout=0.1):
        super().__init__()
        self.d_model = d_model
        self.mult = mult
        self.n_experts = n_experts
        self.dropout = nn.Dropout(dropout)

        H = int(d_model * mult)
        self.router_w = nn.Parameter(torch.empty(d_model, n_experts))
        nn.init.normal_(self.router_w, mean=0.0, std=0.02)

        self.experts = nn.ModuleList([
            nn.Sequential(
                nn.Linear(d_model, H, bias=False),
                nn.SiLU(),
                nn.Linear(H, d_model, bias=False),
            ) for _ in range(n_experts)
        ])

    def forward(self, x):
        """
        x: (B,T,D)
        return: out, aux_loss
        """
        B, T, D = x.shape
        logits = torch.einsum("btd,de->bte", x, self.router_w)   # (B,T,E)
        gates = F.softmax(logits, dim=-1)                        # (B,T,E)
        top1 = torch.argmax(logits, dim=-1)                      # (B,T)

        out = torch.zeros_like(x)
        for e, expert in enumerate(self.experts):
            mask = (top1 == e)                                   # (B,T)
            if not mask.any():
                continue
            xe = x[mask]                                        # (N_e, D)
            ye = expert(xe)                                     # (N_e, D)
            ye = self.dropout(ye)
            out[mask] = ye

        # balance loss: expert_usage ~ 1/n_experts
        expert_usage = gates.mean(dim=(0, 1))                   # (E,)
        aux_loss = torch.sum((expert_usage - 1.0 / self.n_experts) ** 2)
        return out, aux_loss


class DecoderBlockMoEPT(nn.Module):
    """
    对应 Flax DecoderBlockMoE:
    - RMSNorm → SelfAttention → residual
    - RMSNorm → SwitchMoE → residual (+ aux_loss)
    """
    def __init__(self, d_model, n_heads,
                 attn_dropout=0.1,
                 moe_dropout=0.1,
                 moe_mult=2.667,
                 n_experts=8,
                 resid_scale_init=1e-2,
                 max_len=2048):
        super().__init__()
        self.d_model = d_model
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads

        self.res_scale = nn.Parameter(torch.full((), resid_scale_init))

        self.norm1 = RMSNorm(d_model)
        self.qkv_proj = nn.Linear(d_model, 3 * d_model, bias=False)
        self.o_proj = nn.Linear(d_model, d_model, bias=False)
        self.attn_drop = nn.Dropout(attn_dropout)
        self.register_buffer(
            "mask_causal",
            torch.tril(torch.ones(max_len, max_len, dtype=torch.bool)).view(1, 1, max_len, max_len),
            persistent=False,
        )

        self.norm2 = RMSNorm(d_model)
        self.moe = SwitchMoEPT(d_model, mult=moe_mult, n_experts=n_experts, dropout=moe_dropout)

    def _self_attn(self, x):
        B, T, D = x.shape
        H, Dh = self.n_heads, self.head_dim
        qkv = self.qkv_proj(x).view(B, T, 3, H, Dh).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]                      # (B,H,T,Dh)
        att_logits = torch.einsum("bhtd,bhTd->bhtT", q, k) / math.sqrt(Dh)
        mask = self.mask_causal[:, :, :T, :T]                # (1,1,T,T)
        att_logits = att_logits.masked_fill(~mask, float("-inf"))
        att = F.softmax(att_logits, dim=-1)
        att = self.attn_drop(att)
        y = torch.einsum("bhtT,bhTd->bhtd", att, v)
        y = y.permute(0, 2, 1, 3).contiguous().view(B, T, D)
        y = self.o_proj(y)
        return y

    def forward(self, x, mask=None):
        # Self-attention
        h = self.norm1(x)
        att_out = self._self_attn(h)
        x = x + self.res_scale * att_out

        # MoE
        h = self.norm2(x)
        moe_out, aux_loss = self.moe(h)
        x = x + self.res_scale * moe_out
        return x, aux_loss


class MoETransformerPT(nn.Module):
    """
    PyTorch 版 MoETransformer:
    forward(idx, targets=None, pad_mask=None, aux_loss_weight=0.01)
      → logits, total_loss, aux_loss
    """
    def __init__(self, vocab_size, d_model, n_layers, n_heads,
                 n_experts=8, dropout=0.1, moe_mult=2.667, max_len=2048):
        super().__init__()
        self.vocab_size = vocab_size
        self.d_model = d_model
        self.n_layers = n_layers

        self.tok_embed = nn.Embedding(vocab_size, d_model)
        self.dropout = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([
            DecoderBlockMoEPT(
                d_model, n_heads,
                attn_dropout=dropout,
                moe_dropout=dropout,
                moe_mult=moe_mult,
                n_experts=n_experts,
                resid_scale_init=1e-2,
                max_len=max_len,
            )
            for _ in range(n_layers)
        ])
        self.final_norm = RMSNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        self.lm_head.weight = self.tok_embed.weight

    def forward(self, idx, targets=None, pad_mask=None, aux_loss_weight=0.01):
        B, T = idx.shape
        x = self.tok_embed(idx)
        x = self.dropout(x)

        aux_total = 0.0
        for blk in self.blocks:
            x, aux = blk(x)
            aux_total = aux_total + aux

        x = self.final_norm(x)
        logits = torch.einsum("btd,vd->btv", x, self.tok_embed.weight)

        loss = None
        if targets is not None:
            ce = F.cross_entropy(
                logits.view(-1, logits.size(-1)),
                targets.view(-1)
            )
            loss = ce + aux_loss_weight * aux_total
        return logits, loss, aux_total


### Relative Position Bias + Local-Global

In [ ]:
# ========= Relative Position Bias + Local-Global Attention =========

def relative_position_bucket_pt(rel_pos, bidirectional: bool, num_buckets=32, max_distance=128):
    """
    PyTorch 版 relative_position_bucket
    rel_pos: (T_q, T_k) int
    """
    if bidirectional:
        n = num_buckets // 2
    else:
        n = num_buckets
    sign = (rel_pos < 0).int() if bidirectional else 0
    rp = rel_pos.abs()

    max_exact = n // 2
    is_small = rp < max_exact

    # avoid log(0)
    rp_clamped = torch.max(rp, torch.ones_like(rp))
    val_large = max_exact + (
        torch.log(rp_clamped.float() / max_exact) /
        torch.log(torch.tensor(max_distance / max_exact))
    ) * (n - max_exact)
    val_large = val_large.to(torch.int32)
    val_large = torch.minimum(val_large, torch.tensor(n - 1, dtype=torch.int32))

    buckets = torch.where(is_small, rp, val_large)
    return buckets + sign * n


class RelPosBiasPT(nn.Module):
    """
    对应 Flax RelPosBias:
    输出 bias: (1, H, T_q, T_k)
    """
    def __init__(self, num_buckets, num_heads, max_distance, bidirectional=False):
        super().__init__()
        self.num_buckets = num_buckets
        self.num_heads = num_heads
        self.max_distance = max_distance
        self.bidirectional = bidirectional
        total_buckets = num_buckets * (2 if bidirectional else 1)
        self.bias_table = nn.Parameter(torch.empty(total_buckets, num_heads))
        nn.init.normal_(self.bias_table, mean=0.0, std=0.02)

    def forward(self, qlen, klen, device=None):
        device = device or self.bias_table.device
        cp = torch.arange(qlen, device=device)[:, None]    # (T_q,1)
        mp = torch.arange(klen, device=device)[None, :]    # (1,T_k)
        rel = mp - cp                                      # (T_q,T_k)
        buckets = relative_position_bucket_pt(
            rel,
            bidirectional=self.bidirectional,
            num_buckets=self.num_buckets,
            max_distance=self.max_distance
        )                                                 # (T_q,T_k)
        bias = self.bias_table[buckets]                   # (T_q,T_k,H)
        bias = bias.permute(2, 0, 1).unsqueeze(0)         # (1,H,T_q,T_k)
        return bias


class LocalGlobalSelfAttentionPT(nn.Module):
    """
    对应 Flax LocalGlobalSelfAttention:
    - 每个 batch 共享 n_global 个可学习 global tokens
    - Q 来自 x 本身
    - K,V 来自 concat(globals, x)
    - 相对位置 bias 只作用在 local part（非 global）上
    - mask: 每个 token 看所有 globals + local window 内的历史 tokens
    """
    def __init__(self, d_model, n_heads,
                 window=512, n_global=8,
                 attn_dropout=0.1,
                 num_buckets=32,
                 max_distance=512):
        super().__init__()
        self.d_model = d_model
        self.n_heads = n_heads
        self.window = window
        self.n_global = n_global
        self.head_dim = d_model // n_heads

        self.globals = nn.Parameter(torch.empty(n_global, d_model))
        nn.init.normal_(self.globals, mean=0.0, std=0.02)

        self.q_proj = nn.Linear(d_model, d_model, bias=False)
        self.k_proj = nn.Linear(d_model, d_model, bias=False)
        self.v_proj = nn.Linear(d_model, d_model, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)

        self.rp_bias = RelPosBiasPT(num_buckets, n_heads, max_distance, bidirectional=False)
        self.attn_drop = nn.Dropout(attn_dropout)

    def forward(self, x):
        """
        x: (B,T,D)
        """
        B, T, D = x.shape
        H = self.n_heads
        Dh = self.head_dim
        device = x.device

        # global tokens: (B,G,D)
        g = self.globals.to(device).unsqueeze(0).expand(B, self.n_global, D)
        cat = torch.cat([g, x], dim=1)                    # (B,G+T,D)
        G = self.n_global

        # projections
        q = self.q_proj(x).view(B, T, H, Dh).permute(0, 2, 1, 3)           # (B,H,T,Dh)
        k = self.k_proj(cat).view(B, G + T, H, Dh).permute(0, 2, 1, 3)     # (B,H,G+T,Dh)
        v = self.v_proj(cat).view(B, G + T, H, Dh).permute(0, 2, 1, 3)

        # 相对位置 bias：只针对 local-local 部分，形状: (1,H,T,T)
        rp_bias = self.rp_bias(T, T, device=device)                        # (1,H,T,T)
        zero_g = torch.zeros(1, H, T, G, device=device)                    # 对 global 的 bias = 0
        rel_bias = torch.cat([zero_g, rp_bias], dim=-1)                    # (1,H,T,G+T)

        # local window mask
        ar = torch.arange(T, device=device)
        dist = ar[None, :] - ar[:, None]   # (T,T)
        local = (dist >= -self.window) & (dist <= 0)    # 只看历史 + 自己
        mask_seq = local.unsqueeze(0).unsqueeze(0)      # (1,1,T,T)

        # mask: global 全可见 + local-window
        mask = torch.cat([
            torch.ones(1, 1, T, G, dtype=torch.bool, device=device),
            mask_seq
        ], dim=-1)                                      # (1,1,T,G+T)

        # attention
        logits = torch.einsum("bhtd,bhkd->bhtk", q, k) / math.sqrt(Dh)     # (B,H,T,G+T)
        logits = logits + rel_bias                                         # broadcast (1,H,T,G+T)
        logits = logits.masked_fill(~mask, float("-inf"))
        att = F.softmax(logits, dim=-1)
        att = self.attn_drop(att)
        y = torch.einsum("bhtk,bhkd->bhtd", att, v)                        # (B,H,T,Dh)
        y = y.permute(0, 2, 1, 3).contiguous().view(B, T, D)
        y = self.out_proj(y)
        return y


class DecoderBlockLocalGlobalPT(nn.Module):
    """
    对应 Flax DecoderBlockLocalGlobal:
    - RMSNorm → LocalGlobalSelfAttention → residual
    - RMSNorm → SwiGLU → residual
    """
    def __init__(self, d_model, n_heads,
                 window=512, n_global=8,
                 attn_dropout=0.1, mlp_dropout=0.1,
                 resid_scale_init=1e-2):
        super().__init__()
        self.d_model = d_model
        self.res_scale = nn.Parameter(torch.full((), resid_scale_init))
        self.norm1 = RMSNorm(d_model)
        self.norm2 = RMSNorm(d_model)
        self.attn = LocalGlobalSelfAttentionPT(
            d_model, n_heads,
            window=window,
            n_global=n_global,
            attn_dropout=attn_dropout
        )
        self.mlp = SwiGLU(d_model, mult=2.667, dropout=mlp_dropout)

    def forward(self, x):
        h = self.norm1(x)
        h = self.attn(h)
        x = x + self.res_scale * h

        h = self.norm2(x)
        h = self.mlp(h)
        x = x + self.res_scale * h
        return x


class RelPosLocalGlobalTransformerPT(nn.Module):
    """
    PyTorch 版 RelPosLocalGlobalTransformer:
    forward(idx, targets=None) → (logits, loss)
    """
    def __init__(self, vocab_size, d_model, n_layers, n_heads,
                 window=512, n_global=8, dropout=0.1):
        super().__init__()
        self.vocab_size = vocab_size
        self.d_model = d_model
        self.n_layers = n_layers

        self.tok_embed = nn.Embedding(vocab_size, d_model)
        self.dropout = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([
            DecoderBlockLocalGlobalPT(
                d_model, n_heads,
                window=window,
                n_global=n_global,
                attn_dropout=dropout,
                mlp_dropout=dropout,
                resid_scale_init=1e-2
            )
            for _ in range(n_layers)
        ])
        self.final_norm = RMSNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        self.lm_head.weight = self.tok_embed.weight

    def forward(self, idx, targets=None):
        B, T = idx.shape
        x = self.tok_embed(idx)
        x = self.dropout(x)

        for blk in self.blocks:
            x = blk(x)

        x = self.final_norm(x)
        logits = torch.einsum("btd,vd->btv", x, self.tok_embed.weight)

        loss = None
        if targets is not None:
            loss = F.cross_entropy(
                logits.view(-1, logits.size(-1)),
                targets.view(-1)
            )
        return logits, loss
